<span style="color:lightgreen">

# L4.4 Transcripción con Traducción del portugués al inglés en cascada Finetuneando modelo Covost2 pt-en

<span>

# Fine-tuning of the NLLB model for Europarl-ST (Spanish to English)

In this notebook, we are going to explain how to fine-tune the [NLLB model](https://huggingface.co/docs/transformers/model_doc/nllb) on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST). First, we need to run the experiment L4.1 on the development set to generate data to be used in the fine tuning process. Once the fine tuning of the NLLB is carried out, we will evaluate the NLLB model performance over the Europarl-ST test data where speech transcriptions were obtained also in the L4.1 experiment. Remember that the initial performance of the NLLB model on the test data evaluated in experiment L4.3 was 9.3 of BLEU and 57.9% COMET.

We load the automatic speech transcriptions and reference translations from the csv file generated in experiment L4.1 on the dev data. Moreover, we split the data into train (17%) and validation (test) (83%) sets for fine tuning

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration(
    max_epoch=5
)

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks_pt_en
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
from datasets import load_dataset

# NOTE: L4,1 tine datos de transcripción de TRAIN
# Cuando lo partimos la partición se llama test pero no es validation
raw_datasets = load_dataset("csv", data_files=os.path.join(CONFIG.RESULTS_PATH, "L4.1_ASR_Whisper_Baseline_Covost2_pt-en-train.csv"))

raw_datasets = raw_datasets.select_columns(["hypothesis","translation"])

raw_datasets = raw_datasets["train"].train_test_split(test_size=0.83, shuffle=False)

# Filter out rows with null values in hypothesis or translation
raw_datasets = raw_datasets.filter(lambda x: x["hypothesis"] is not None and x["translation"] is not None)

print(raw_datasets)

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/170 [00:00<?, ? examples/s]

Filter:   0%|          | 0/830 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['hypothesis', 'translation'],
        num_rows: 170
    })
    test: Dataset({
        features: ['hypothesis', 'translation'],
        num_rows: 830
    })
})


Show the training pairs (hypothesis-translation)

In [3]:
raw_datasets["train"][:12]["hypothesis"]

[' Sua alma deve ser muito primitiva para entender essas coisas.',
 ' A tristeza em casa é um desejo mórbido de voltar para casa.',
 ' O nome original em inglês deste livro não é tão sensacional.',
 ' Não, não obrigado, apenas procurando.',
 ' Essa palavra é uma sigua?',
 ' Voltpece informações.',
 ' Lara Gosse de Laranjas',
 ' Eles entraram da chuva toda desgrenhada e fumigante.',
 ' Um dançarino está realizando um movimento de dança.',
 ' São cerca de cinco.',
 ' Os pd-serizando em torno de carro em uma rua movimentada.',
 ' Ela se sentou no chão tomando seu café.']

In [4]:
raw_datasets["train"][:12]["translation"]

['YourHis/Her soul must be too primitive to understand these things.',
 'Homesickness is a morbid desire to return home.',
 'The original name in English of this book is not so impressive.',
 'No, no, thanks, just looking.',
 'Is this word an acronym?',
 'Come back and ask for information.',
 'Lara likes oranges.',
 'When they came out of the rain, they were all disheveled and wet.',
 'A dancer is performing a dance move.',
 "It's about five.",
 'Pedestrians walking around cars on a busy street.',
 'She sat on the floor and drank her coffee.']

<p style="page-break-after:always;"></p>

Now we load the pre-trained tokenizer for the NLLB model and apply it to the Spanish-English pair:

In [5]:
max_tok_length = 275

from transformers import AutoTokenizer

checkpoint = "facebook/nllb-200-distilled-600M"
# from flores200_codes import flores_codes
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint, 
    padding=True, 
    pad_to_multiple_of=8, 
    src_lang=CONFIG.src_code, 
    tgt_lang=CONFIG.tgt_code, 
    truncation=True, 
    max_length=max_tok_length,
    )

We can apply the tokenizer function to any dataset taking advantage that Hugging Face Datasets are [Apache Arrow](https://arrow.apache.org) files stored on the disk, so you only keep the samples you ask for loaded in memory.

To keep the data as a dataset, we will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the training needs of the model that is to be finetuned:

In [6]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["hypothesis"], 
        text_target = sample["translation"],
        truncation=True,
        max_length=max_tok_length
    )
    return model_inputs

<p style="page-break-after:always;"></p>

Now, we can apply the preprocess_function to the raw datasets (training, validation and test):

In [7]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/170 [00:00<?, ? examples/s]

Map:   0%|          | 0/830 [00:00<?, ? examples/s]

We can take a quick look at the length histogram in the source language:

In [8]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

 5   4
 6   6
 7   8
 8  12
 9  14
10  16
11  19
12  15
13  10
14   9
15   8
16   7
17  14
18   8
19   7
21   6
22   5
23   1
24   1


Checking a sample after filtering by maximum number of tokens:

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>


In [9]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [10]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config
    )


Next, you should call the prepare_model_for_kbit_training() function to preprocess the quantized model for training.

In [11]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False, gradient_checkpointing_kwargs={'use_reentrant':False})

[LoRA (Low-Rank Adaptation of Large Language Models)](https://huggingface.co/docs/peft/task_guides/lora_based_methods) is a [parameter-efficient fine-tuning (PEFT)](https://huggingface.co/docs/peft/index) technique that significantly reduces the number of trainable parameters. It works by inserting a smaller number of new weights into the model and only these are trained. This makes training with LoRA much faster, memory-efficient, and produces smaller model weights (a few hundred MBs), which are easier to store and share.

<p style="page-break-after:always;"></p>

Each PEFT method is defined by a PeftConfig class that stores all the important parameters for building a PeftModel. For example, to train with LoRA, load and create a LoraConfig class and specify the following parameters:

<ul>
<li>task_type: the task to train for (sequence-to-sequence language modeling in this case)</li>
<li>r: the dimension of the low-rank matrices</li>
<li>lora_alpha: the scaling factor for the low-rank matrices</li>
<li>target_modules: determine what set of parameters are adapted</li>
<li>lora_dropout: the dropout probability of the LoRA layers</li>
</ul>

In [12]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    task_type="SEQ_2_SEQ_LM",
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

Once LoRA and the quantization are setup, create a quantized PeftModel with the get_peft_model() function. It takes a quantized model and the LoraConfig containing the parameters for how to configure a model for training with LoRA.

In [13]:
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


The function that is responsible for putting together samples inside a batch is called a collate function. It is an argument you can pass when you build a DataLoader, the default being a function that will just convert your samples to PyTorch tensors and concatenate them. This is not possible in our case since the inputs we have are not all of the same size. We have deliberately postponed the padding, to only apply it as necessary on each batch and avoid having over-long inputs with a lot of padding.

To do this in practice, we have to define a collate function that will apply the correct amount of padding to the items of the dataset we want to batch together. Fortunately, the Transformers library provides us with such a function via DataCollatorForSeq2Seq that takes a tokenizer when you instantiate it (to know which padding token to use, and whether the model expects padding to be on the left or on the right of the inputs), so we will also need to instantiate the model first to provide it to the collate function:

In [14]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, 
    model=lora_model, 
    pad_to_multiple_of=8
)

## Evaluation

The last thing to define for our Seq2SeqTrainer is how to compute the metrics to evaluate the predictions of our model with respect to references. To this purpose, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu). You can see a simple example of usage below:

In [15]:
from evaluate import load

metric = load("sacrebleu")

<p style="page-break-after:always;"></p>

We need to define a function compute_metrics to compute BLEU scores at each epoch. The example below performs a basic post-processing to decode the predictions into texts:

In [16]:
import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    #labels = np.where(labels < 0, labels, tokenizer.pad_token_id)
    for i in range(len(labels)):
        labels[i] = [tokenizer.pad_token_id if j<0 else j for j in labels[i]]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

<p style="page-break-after:always;"></p>

## Training

The first step before we can define our [Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer#trainer) is to define a [Seq2SeqTrainingArguments class](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Seq2SeqTrainingArguments) that will contain all the hyperparameters the Trainer will use for training and evaluation. The only compulsory argument you have to provide is a directory where the trained model will be saved, as well as the checkpoints along the way. For all the rest, you can set them depending on the recommendations from the model developers:

In [17]:
from transformers import Seq2SeqTrainingArguments

batch_size = 2
model_name = checkpoint.split("/")[-1]
args = Seq2SeqTrainingArguments(
    os.path.join(CONFIG.MODELS_PATH, CONFIG.fine_tune_model_name),
    eval_strategy = "epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=CONFIG.max_epoch,
    predict_with_generate=True,
)

Once we have our model, we can define a Trainer by passing it all the objects constructed up to now — the model, the training_args, the training and validation datasets, the tokenizer, the data collator and the compute_metrics function:

In [18]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    lora_model,
    args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


/tmp/ipykernel_319309/179117317.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


To fine-tune the model on our dataset, we just have to call the [train() function](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Trainer.train) of our Trainer:

In [19]:

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,No log,1.329095,38.416000,13.162700
2,No log,1.258791,39.627300,12.963900
3,No log,1.222636,40.222100,12.907200
4,No log,1.203729,40.381400,12.875900
5,No log,1.198250,40.555500,12.860200


TrainOutput(global_step=425, training_loss=1.1987334127987133, metrics={'train_runtime': 395.5791, 'train_samples_per_second': 2.149, 'train_steps_per_second': 1.074, 'total_flos': 35041951875072.0, 'train_loss': 1.1987334127987133, 'epoch': 5.0})

## Evaluation of the fine tuned NLLL model on the Europarl-ST test set

Let us first load the default inference parameters of NLLB: 

In [20]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "eos_token_id": 2,
  "max_length": 200,
  "pad_token_id": 1
}



We load the speech transcriptions (and reference translations) of Europarl-ST test set from the csv file generated in the experiment L4.1. Then, we prepare the test set in batches to be translated:

In [21]:
from datasets import load_dataset

# NOTE: L4,1 tine datos de transcripción de test
# Cuando lo cargamos la partición se llama train pero no es train es test
raw_datasets = load_dataset("csv", data_files=os.path.join(CONFIG.RESULTS_PATH, "L4.1_ASR_Whisper_Baseline_Covost2_pt-en.csv"))
raw_datasets = raw_datasets.filter(lambda x: x["hypothesis"] is not None and x["translation"] is not None)

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'hypothesis', 'reference', 'translation', 'hypothesis_clean', 'reference_clean', 'translation_clean'],
        num_rows: 4023
    })
})


Each sample pair is preprocessed according to the training needs of the model that has been finetuned:

In [22]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["hypothesis"], 
        text_target = sample["translation"],
        truncation=True,
        max_length=max_tok_length
        )
    return model_inputs

In [23]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/4023 [00:00<?, ? examples/s]

In [24]:
test_batch_size = 32
# NOTE: L4,1 tine datos de transcripción de test
# Cuando lo cargamos la partición se llama train pero no es train es test
batch_tokenized_test = tokenized_datasets['train'].batch(test_batch_size)

Batching examples:   0%|          | 0/4023 [00:00<?, ? examples/s]

Processing in batches to add padding and converting to tensors, then perform inference with num_beams = 1 and do_sample = False, that is, greedy search.

In [25]:
number_of_batches = len(batch_tokenized_test["hypothesis"])
output_sequences = []
for i in range(number_of_batches):
    inputs = tokenizer(batch_tokenized_test["hypothesis"][i], max_length=max_tok_length, truncation=False, return_tensors="pt", padding=True)
    output_batch = lora_model.generate(generation_config=generation_config, input_ids=inputs["input_ids"].cuda(), attention_mask=inputs["attention_mask"].cuda(), forced_bos_token_id=tokenizer.convert_tokens_to_ids(CONFIG.tgt_code), max_length = max_tok_length, num_beams=1, do_sample=False,)
    output_sequences.extend(output_batch.cpu())

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2914: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


We can recover the decoded predictions and references by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [26]:
decoded_preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)

In [27]:
# NOTE: L4,1 tine datos de transcripción de test
# Cuando lo cargamos la partición se llama train pero no es train es test
references = tokenizer.batch_decode(tokenized_datasets["train"]["labels"], skip_special_tokens=True)

Let's take a closer look at some decoded predictions and references:

In [28]:
references[:5]

['Borrow money from people in the village',
 'Lock them up',
 'Youtube is still the best video platform.',
 'A girl and a boy kissing in the shadows',
 'The wizard cast a very powerful spell on the city.']

In [29]:
decoded_preds[:5]

['And the day of borrowing people will be borrowing.',
 'Get your asses off.',
 'YouTube is still the best video platform.',
 'Girl and boy, peeping in the shadows.',
 'The magician showed a very powerful spell in the city.']

Predictions and references are normalized using the Whisper basic text standardisation/normalization module

In [30]:
from whisper.normalizers.basic import BasicTextNormalizer

normalizer = BasicTextNormalizer()

decoded_preds_clean = [normalizer(text) for text in decoded_preds]
references_clean = [normalizer(text) for text in references]

In [31]:
from evaluate import load

metric = load("sacrebleu")

For evaluation, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu).

In [32]:
result = metric.compute(predictions=decoded_preds_clean, references=references_clean)
print(f'BLEU score: {result["score"]:.1f}')

BLEU score: 37.4


In [33]:
from evaluate import load
comet_metric = load('comet')

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


Encoder model frozen.


/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


Compute COMET figures using the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics.

In [34]:
comet_score = comet_metric.compute(predictions=decoded_preds_clean, references=references_clean, sources=raw_datasets["train"]["hypothesis"])

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [35]:
print(f"COMET: {comet_score['mean_score'] * 100:.2f} %")

COMET: 76.28 %


As a a result of the fine tuning, we have achieved a relative improvement of 173% in terms of BLEU (from 9.3 to 25.4) and about 21% in terms of COMET (from 57.9% to 70.1%)

# Exercise

Perform a similar experiment using the Covost2 source-english setup previously used in L4.1. Evaluate the performance using the best whisper models obtained in L4.1. 